# 03 · Explaining the forecast with SHAP

TreeSHAP via LightGBM `pred_contrib`. With the anchored target, contributions explain the deviation from the previous day's mean price:

forecast = prev-day mean + base + Σ SHAP

In [1]:
import numpy as np
import plotly.io as pio
import polars as pl

from energy_forecast import explain, features, forecast, plots
from energy_forecast.data import load

pio.templates.default = "plotly_white"
df = features.build(load())
cfg = forecast.DEFAULT
model = forecast.fit(forecast.train_window(df, forecast.TEST[0], cfg), cfg)
test = df.filter(pl.col("day").is_between(*forecast.TEST))
shap = explain.shap_values(model, test, cfg)

In [2]:
total = shap.select(pl.sum_horizontal(*cfg.features, "base", "anchor")).to_series().to_numpy()
np.abs(total - forecast.predict(model, test, cfg)).max()

np.float64(1.7479351299698465e-12)

## Global importance

In [3]:
imp = explain.importance(shap, cfg)
plots.shap_importance(imp)

## Dependence

In [4]:
for f in imp["feature"].head(4):
    plots.shap_dependence(test, shap, f).show()

## Single days: cheapest and most expensive day of the test year

In [5]:
daily = test.group_by("day").agg(pl.col("price").mean()).sort("price")
cheap, expensive = daily["day"][0], daily["day"][-1]
cheap, expensive

(datetime.date(2026, 4, 5), datetime.date(2026, 10, 29))

In [6]:
plots.shap_day(shap, list(cfg.features), cheap)

In [7]:
plots.shap_day(shap, list(cfg.features), expensive)